# Time optimization of the app development process

IS6003 final assignment. Research method: DSRM (Peffers et al., 2007). Process analysis follows *Fundamentals of BPM*, 2nd ed. (Dumas et al.).

**Status: walking skeleton.** This runs the whole path on the corrected as-is model (`models/as_is.bpmn`, changes in `models/as_is_changelog.md`): BPMN file → simulation → event log → baseline metrics → `web/results.json`. The parameters are placeholders (fixed durations, every case at time 0, no capacity limits) until ticket 10, so the numbers mean nothing yet. Later tickets add the real parameters, the analysis, ML and scenarios.

In [ ]:
from pathlib import Path

from bpmopt.analysis import baseline_metrics, flow_analysis_check
from bpmopt.bpmn import read_bpmn
from bpmopt.parameters import load_parameters
from bpmopt.results import build_results, export_results
from bpmopt.simulation import simulate
from bpmopt.value_added import load_value_added

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MODEL = ROOT / "models/as_is.bpmn"
PARAMETERS = ROOT / "models/as_is.params.toml"
VALUE_ADDED = ROOT / "models/as_is.value_added.toml"
SEED = 42
N_CASES = 100

## Simulate the model

The simulator executes the BPMN file directly, so the diagram is exactly the process that is simulated. Each row of the event log is one element execution (book §11.3).

In [ ]:
model = read_bpmn(MODEL)
params = load_parameters(PARAMETERS)
log = simulate(model, params, n_cases=N_CASES, seed=SEED)
log.head(10)

## Baseline

From the event log: cycle, processing and waiting time per case (mean, median, P90), cycle time efficiency (§7.1.2, Eq. 7.5), the rework rate per rework gateway and the mean waiting time per activity. Processing time counts the time some task of the case is in progress, so parallel work counts once, like the theoretical cycle time (p.262). Rejected proposals end early, so they count with a short cycle time.

In [ ]:
baseline_metrics(log, model)

## Flow analysis check

The book's flow analysis (§7.1.1) ignores resource contention (p.273), so the model is simulated again with unlimited capacity. The two means should agree within sampling error.

In [ ]:
flow_analysis_check(model, params, n_cases=N_CASES, seed=SEED)

## Export results for the page

The value-added class of each task (book §6.1) comes from `models/as_is.value_added.toml`. It is a draft awaiting review, see `docs/analysis/phase1-qualitative.md`.

In [ ]:
results = build_results(log=log, model=model, params=params, seed=SEED)
results["value_added"] = load_value_added(VALUE_ADDED, model)
export_results(results, ROOT / "web", models={"as_is": MODEL})